In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, CSVLogger, EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras import layers, models, Input
from tqdm import tqdm
import gc 

# Suas bibliotecas customizadas
import utils.processamento_dados as proc_dados
import utils.metricas_e_visualizacao as met_vil

In [ ]:
# Configuração de GPU
from tensorflow.keras import mixed_precision
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU habilitada com sucesso!")
    except RuntimeError as e:
        print(e)

mixed_precision.set_global_policy("mixed_float16")
tf.get_logger().setLevel('ERROR')

In [ ]:
# # ==========================================
# # 1. DEFINIÇÃO DO MODELO
# # ==========================================
# def create_model_3d(input_shape, n_classes):
#     inputs = Input(shape=input_shape)

#     # Camada 1
#     x = layers.Conv3D(4, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(inputs)
#     x = layers.BatchNormalization()(x)
#     x = layers.LeakyReLU(negative_slope=0.3)(x)
#     x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
#     x = layers.Dropout(0.3)(x)

#     # Camada 2
#     x = layers.Conv3D(8, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.LeakyReLU(negative_slope=0.3)(x)
#     x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
#     x = layers.Dropout(0.3)(x)

#     # Camada 3
#     x = layers.Conv3D(16, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.LeakyReLU(negative_slope=0.3)(x)
#     x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
#     x = layers.Dropout(0.3)(x)

#     # Classificador
#     x = layers.Flatten()(x)
#     x = layers.Dense(16, kernel_regularizer=l2(0.01))(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.Dropout(0.3)(x)
#     x = layers.LeakyReLU(negative_slope=0.3)(x)

#     if n_classes == 2:
#         outputs = layers.Dense(1, activation='sigmoid')(x)
#     else:
#         outputs = layers.Dense(n_classes, activation='softmax')(x)

#     model = models.Model(inputs=inputs, outputs=outputs, name="Small_Leaky_3D_Model")
#     return model


In [ ]:
def create_model_3d(input_shape, n_classes):
    # Entrada
    inputs = Input(shape=input_shape)  # (D, H, W, C)

    # === Camada 1 ===
    x = layers.Conv3D(4, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(inputs)
    x = layers.Conv3D(4, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(4, 4, 4), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # === Camada 2 ===
    x = layers.Conv3D(8, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.Conv3D(8, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # === Camada 3 ===
    x = layers.Conv3D(16, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.Conv3D(16, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # === Classificador ===
    x = layers.Flatten()(x)

    x = layers.Dense(16, kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)

    # === Adaptação da Saída ===
    if n_classes == 2:
        # Binário: 1 neurônio com Sigmoid (para funcionar com binary_crossentropy)
        outputs = layers.Dense(1, activation='sigmoid')(x)
    else:
        # Multiclasse: N neurônios com Softmax
        outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="Small_Leaky_3D_Model")
    
    return model

In [ ]:
# ==========================================
# 2. CONFIGURAÇÕES E CAMINHOS
# ==========================================
dir_base = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI"
data_3t_dir = f"{dir_base}/ADNI_3_4_NORMALIZED"

# Pasta de resultados K-FOLD
results_dir_base = f'{dir_base}/results_only_3t_kfold'
os.makedirs(results_dir_base, exist_ok=True)

n = len(os.listdir(results_dir_base))
folder_name = f"kfold_experiment_{n+1}"
results_dir = os.path.join(results_dir_base, folder_name)
os.makedirs(results_dir, exist_ok=True)
print(f"Resultados serão salvos em: {results_dir}")

adni_class_names = ['cn', 'ad']

In [ ]:
import os
import numpy as np
import nibabel as nib
import gc
import sys

# --- 1. CONFIGURAÇÃO DE CAMINHOS ---
dir_train = os.path.join(data_3t_dir, "train")
dir_val = os.path.join(data_3t_dir, "validation")
dir_test = os.path.join(data_3t_dir, "test")

# Mapeamento de classes (Ajuste conforme suas pastas reais)
class_map = {
    "cn": 0,  # Controle Normal
    "ad": 1  # Alzheimer
}

print("=== ETAPA 1: INDEXANDO ARQUIVOS (SALVANDO CAMINHOS) ===")

all_file_paths = []
all_labels = []

# Função para varrer pastas recursivamente
def indexar_arquivos(diretorio_base):
    caminhos = []
    labels = []
    
    # Walk percorre todas as subpastas
    for root, dirs, files in os.walk(diretorio_base):
        for file in files:
            if file.endswith(".nii") or file.endswith(".nii.gz"):
                full_path = os.path.join(root, file)
                
                # Tenta descobrir o label pelo nome da pasta pai
                folder_name = os.path.basename(root)
                
                # Verifica se a pasta corresponde a uma classe conhecida
                label = None
                for key, val in class_map.items():
                    if key in folder_name: # Ex: se "CN" está em "train/CN"
                        label = val
                        break
                
                if label is not None:
                    caminhos.append(full_path)
                    labels.append(label)
                else:
                    # Opcional: printar aviso se achar arquivo em pasta desconhecida
                    pass
                    
    return caminhos, labels

# Coleta os caminhos
paths_train, y_train_list = indexar_arquivos(dir_train)
paths_val, y_val_list = indexar_arquivos(dir_val)
paths_test, y_test_list = indexar_arquivos(dir_test)

# Junta tudo em uma lista única (se quiser treinar com tudo) 
# OU mantenha separado. Aqui vou juntar tudo conforme seu pedido anterior de X_all.
final_paths = paths_train + paths_val + paths_test
final_labels = y_train_list + y_val_list + y_test_list

total_files = len(final_paths)
print(f"Indexação concluída.")
print(f"Total de arquivos encontrados: {total_files}")
print(f"Exemplo de caminho: {final_paths[0]}")
print(f"Exemplo de label: {final_labels[0]}")

if total_files == 0:
    print("Erro: Nenhum arquivo encontrado. Verifique os caminhos e o 'class_map'.")
    sys.exit(1)


print("\n=== ETAPA 2: PREPARANDO MEMÓRIA ===")

# Descobre o shape carregando apenas o primeiro arquivo
try:
    print("🔍 Lendo primeiro arquivo para pegar dimensões...")
    img_obj = nib.load(final_paths[0])
    img_data = img_obj.get_fdata()
    sample_shape = img_data.shape
    
    # Normalização básica (opcional - deve ser igual ao que você usava)
    # Exemplo: normalizar entre 0 e 1 se necessário
    # img_data = (img_data - np.min(img_data)) / (np.max(img_data) - np.min(img_data))
    
    print(f"Dimensões detectadas: {sample_shape}")
    print(f"Tipo de dado original: {img_data.dtype}")
    
except Exception as e:
    print(f"Erro ao ler arquivo de amostra: {e}")
    sys.exit(1)

# Aloca o Array Gigante
try:
    # Usamos float32 para economizar RAM
    X_all = np.zeros((total_files, *sample_shape), dtype=np.float32)
    y_all = np.array(final_labels, dtype=np.int32)
    
    gb_use = X_all.nbytes / (1024**3)
    print(f"Memória alocada: {gb_use:.2f} GB")
    
except MemoryError:
    print("ERRO FATAL: Falta RAM para criar X_all.")
    sys.exit(1)


print("\n=== ETAPA 3: CARREGANDO DADOS (LOOP FINAL) ===")

# Loop pelos caminhos para carregar os dados
for i, file_path in enumerate(final_paths):
    try:
        # 1. Carrega NIfTI
        img = nib.load(file_path)
        data = img.get_fdata() # Retorna float64 por padrão
        
        # Exemplo de conversão para float32
        data = data.astype(np.float32)
        
        # 3. Insere no array
        X_all[i] = data
        
        # Log de progresso a cada 100 imagens
        if i % 100 == 0:
            print(f"  > Processado {i}/{total_files}...")
            
    except Exception as e:
        print(f"Erro ao ler arquivo {file_path}: {e}")
        # Opcional: zerar essa posição ou tratar depois

print("\n=== FINALIZADO ===")
print(f"Shape final X_all: {X_all.shape}")
print(f"Shape final y_all: {y_all.shape}")

# Limpeza final
del final_paths, final_labels
gc.collect()

In [ ]:
K_FOLDS=5
BATCH_SIZE=8
EPOCHS=50

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.model_selection import StratifiedKFold
import numpy as np
import gc
import os

# ==========================================
# CONFIGURAÇÕES GERAIS
# ==========================================
K_FOLDS = 5
BATCH_SIZE = 8       # Mantendo baixo para 3D
LEARNING_RATE = 1e-4 # LR menor para Fine-Tuning
EPOCHS = 50

# Listas globais para resultados
fold_accuracies = []
fold_aucs = []
aggregated_true_labels = []
aggregated_pred_labels = []

# ==========================================
# FUNÇÕES AUXILIARES (Balanceamento e Generators)
# ==========================================

def augment_and_balance_in_ram(X_full, y_full, indices):
    """
    Gera o dataset aumentado DIRETO NA RAM, focando na classe minoritária.
    Usa float16 para economizar memória (crucial para 3D).
    """
    # 1. Pega labels deste fold para contagem
    y_subset = y_full[indices]
    
    # 2. Identifica classe minoritária
    counts = np.bincount(y_subset)
    minority_class = np.argmin(counts)
    majority_class = np.argmax(counts)
    n_minority = counts[minority_class]
    n_majority = counts[majority_class]
    
    print(f"  Balanceamento: Majoritária ({majority_class}): {n_majority} | Minoritária ({minority_class}): {n_minority}")
    
    # 3. Calcula tamanho final (Majoritária + 4x Minoritária)
    # Ajuste o multiplicador se necessário
    total_final = n_majority + (n_minority * 4)
    
    input_shape = X_full[0].shape
    # Adiciona canal extra se não existir (D, H, W, 1)
    final_shape = (total_final, *input_shape, 1)
    
    print(f"  Alocando memória para {total_final} amostras...")
    
    # 4. Alocação (Float16 é o segredo para não estourar RAM)
    try:
        X_aug = np.empty(final_shape, dtype=np.float16) 
        y_aug = np.empty((total_final,), dtype=np.int32)
    except MemoryError:
        print("  ERRO FATAL: Memória RAM insuficiente para Augmentation.")
        raise
        
    # 5. Preenchimento e Augmentation
    cursor = 0
    total_indices = len(indices)
    
    for idx, i in enumerate(indices):
        orig = X_full[i] 
        lbl = y_full[i]
        
        # Garante dimensão do canal
        if len(orig.shape) == 3:
            orig_expanded = np.expand_dims(orig, axis=-1)
        else:
            orig_expanded = orig

        if lbl == majority_class:
            X_aug[cursor] = orig_expanded
            y_aug[cursor] = lbl
            cursor += 1
        else:
            # Estratégia para Classe Minoritária: Original + 3 Variações
            # 1. Original
            X_aug[cursor] = orig_expanded
            y_aug[cursor] = lbl
            cursor += 1
            
            # 2. Zoom
            zoom_img = proc_dados.augment_zoom(orig)
            X_aug[cursor] = np.expand_dims(zoom_img, axis=-1)
            y_aug[cursor] = lbl
            cursor += 1
            
            # 3. Shift
            shift_img = proc_dados.augment_shift(orig)
            X_aug[cursor] = np.expand_dims(shift_img, axis=-1)
            y_aug[cursor] = lbl
            cursor += 1
            
            # 4. Rotation
            rot_img = proc_dados.augment_rotation(orig)
            X_aug[cursor] = np.expand_dims(rot_img, axis=-1)
            y_aug[cursor] = lbl
            cursor += 1
            
        if idx % 100 == 0:
            print(f"    Processando: {idx}/{total_indices}...", end='\r')
            
    print(f"\n Dados Augmentados na RAM! Shape: {X_aug.shape}")
    
    # 6. Embaralhar os dados gerados
    shuf_idxs = np.arange(total_final)
    np.random.shuffle(shuf_idxs)
    
    return X_aug[shuf_idxs], y_aug[shuf_idxs]

def numpy_generator(x_data, y_data):
    """Generator simples que lê do array NumPy e entrega pro TensorFlow"""
    for i in range(len(x_data)):
        yield x_data[i], y_data[i]

# ==========================================
# LOOP PRINCIPAL (FINE-TUNING K-FOLD)
# ==========================================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

# Nota: Usamos X_3t_all e y_3t_all definidos anteriormente (dataset 3T)
for fold_idx, (train_index, val_index) in enumerate(skf.split(X_all, y_all)):
    print(f"\n{'='*40}")
    print(f"INICIANDO FOLD {fold_idx+1}/{K_FOLDS} (FINE-TUNING)")
    print(f"{'='*40}")

    fold_dir = os.path.join(results_dir, f"fold_ft_{fold_idx+1}")
    os.makedirs(fold_dir, exist_ok=True)

    # --- 1. PREPARAÇÃO DOS DADOS ---
    
    # Validação: Apenas expand dims e cast para float32 (Generator cuidará do resto)
    # Usamos slice numpy normal pois validação é pequena
    X_val_fold = X_all[val_index]
    if len(X_val_fold.shape) == 3:
        X_val_fold = np.expand_dims(X_val_fold, axis=-1)
    X_val_fold = X_val_fold.astype(np.float32)
    y_val_fold = y_all[val_index]

    # Treino: AUGMENTATION NA RAM (Aqui acontece a mágica)
    print("⏳ Gerando Augmentation e Balanceamento na RAM...")
    try:
        # Passamos o dataset inteiro e os índices de treino
        X_train_aug, y_train_aug = augment_and_balance_in_ram(X_all, y_all, train_index)
    except MemoryError:
        print("FATAL: Memória insuficiente. Tentando reduzir augmentation...")
        break

    # --- 2. CONFIGURAR GENERATORS (TF.DATA) ---
    print("Configurando tf.data Generators...")
    
    # Especificações (atenção ao float16 do treino)
    train_spec_img = tf.TensorSpec(shape=X_train_aug.shape[1:], dtype=tf.float16)
    train_spec_lbl = tf.TensorSpec(shape=(), dtype=tf.int32)
    
    val_spec_img = tf.TensorSpec(shape=X_val_fold.shape[1:], dtype=tf.float32)
    val_spec_lbl = tf.TensorSpec(shape=(), dtype=tf.int32)

    # Dataset Treino
    train_dataset = tf.data.Dataset.from_generator(
        lambda: numpy_generator(X_train_aug, y_train_aug),
        output_signature=(train_spec_img, train_spec_lbl)
    )
    train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # Dataset Validação
    val_dataset = tf.data.Dataset.from_generator(
        lambda: numpy_generator(X_val_fold, y_val_fold),
        output_signature=(val_spec_img, val_spec_lbl)
    )
    val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # --- 3. MODELO ---
    input_shape = X_train_aug.shape[1:] # (156, 195, 160, 1)
    
    # Cria arquitetura
    model = create_model_3d(input_shape, 2)

    # Compila (LR Baixo)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    # Callbacks
    model_name = f"model_ft_fold_{fold_idx+1}.keras"
    model_save_path = os.path.join(fold_dir, model_name)
    
    checkpoint = ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=0)
    early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    csv_log = CSVLogger(os.path.join(fold_dir, 'training_log.csv'))

    # --- 4. TREINAMENTO ---
    print("🚀 Iniciando Fine-Tuning...")
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=[checkpoint, early_stop, reduce_lr, csv_log],
        verbose=1
    )
    
    # Salva gráfico histórico
    try: met_vil.plot_training_history_binary(history, fold_dir)
    except: pass

    # --- 5. AVALIAÇÃO E PREDIÇÃO ---
    print("Carregando melhor modelo do fold...")
    best_model = models.load_model(model_save_path)
    
    # Avaliação numérica
    loss, acc, auc = best_model.evaluate(val_dataset, verbose=0)
    fold_accuracies.append(acc)
    fold_aucs.append(auc)
    print(f"  Resultado Fold {fold_idx+1}: Acc={acc:.4f}, AUC={auc:.4f}")

    # Predição (Usando função utilitária ou generator manual se preferir)
    # Como X_val_fold está na RAM e é pequeno comparado ao treino aumentado, podemos passar direto ou em batches
    # Vamos usar get_predictions_binary para consistência
    pred_labels, true_labels, preds_prob = met_vil.get_predictions_binary(X_val_fold, y_val_fold, BATCH_SIZE, best_model)
    
    # Relatórios
    met_vil.get_classification_report(true_labels, pred_labels, fold_dir, f'report_ft_fold_{fold_idx+1}')
    met_vil.plot_confusion_matrix(true_labels, pred_labels, fold_dir, f'conf_matrix_ft_fold_{fold_idx+1}', adni_class_names)

    # Acumula para relatório global
    aggregated_true_labels.extend(true_labels)
    aggregated_pred_labels.extend(pred_labels)

    # --- LIMPEZA DE MEMÓRIA ---
    print("🧹 Limpando memória...")
    del X_train_aug, y_train_aug, X_val_fold, y_val_fold
    del train_dataset, val_dataset, model_ft, best_model, history
    tf.keras.backend.clear_session()
    gc.collect()

In [ ]:
# ==========================================
# 5. RESULTADOS FINAIS E MATRIZ AGREGADA
# ==========================================
print("\n" + "="*40)
print("GERANDO RESULTADOS AGREGADOS")
print("="*40)

# Converter listas acumuladas para numpy arrays para compatibilidade
final_true = np.array(aggregated_true_labels)
final_pred = np.array(aggregated_pred_labels)

# ### NOVO: Gerar Matriz de Confusão Agregada
print("Gerando Matriz de Confusão Agregada...")
met_vil.plot_confusion_matrix(
    final_true, 
    final_pred, 
    results_dir,  # Salva na pasta raiz do experimento (não dentro de fold_X)
    'AGGREGATED_CONFUSION_MATRIX', 
    adni_class_names
)

# ### NOVO: Relatório de Classificação Agregado
print("Gerando Relatório de Classificação Agregado...")
met_vil.get_classification_report(
    final_true, 
    final_pred, 
    results_dir, 
    'AGGREGATED_CLASSIFICATION_REPORT'
)

print(f"\nMédia Acurácia: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
print(f"Média AUC: {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")

# Salvar resumo final em texto
with open(os.path.join(results_dir, "final_kfold_summary.txt"), "w") as f:
    f.write(f"Mean Accuracy: {np.mean(fold_accuracies):.4f} +/- {np.std(fold_accuracies):.4f}\n")
    f.write(f"Mean AUC: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}\n")
    f.write(f"Fold Accuracies: {fold_accuracies}\n")
    f.write(f"\n--- Aggregated Metrics ---\n")
    # Podemos adicionar métricas simples calculadas sobre o total
    from sklearn.metrics import accuracy_score
    total_acc = accuracy_score(final_true, final_pred)
    f.write(f"Total Aggregated Accuracy: {total_acc:.4f}\n")